# 02a — Vectorized phenotype extraction

## Purpose
Extract configured clinical phenotypes from problems, diagnoses, and prescribing data, evaluate their temporal windows, aggregate composite phenotypes, and save reusable parquet artifacts.

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
import os
import pandas as pd
from pathlib import Path

from icare_risk.utils.paths import (
    get_path, get_data_root_from_schema,
    print_paths_summary, print_directory_content
)
from icare_risk.utils.io import (
    load_pkg_yaml, load_yaml_file
)
from icare_risk.utils.diagnostics import (
    check_df, check_value_counts, check_merge_keys,
)
from icare_risk.clinphen.utils.vectorized import (
    run_vectorized,
    evaluate_temporal_windows,
    aggregate_composite_phenotypes,
    parse_phenotypes_to_batches
)

# Paths
DATA_DIR = get_path('data/synthetic/10000p')
DATA_DIR = get_data_root_from_schema() 
DATA_PROC_DIR = DATA_DIR / 'processed'

print("Current Working Directory:", os.getcwd())
print("Data Path:", DATA_DIR)
print("Data Processed Path:", DATA_PROC_DIR)
paths_map = print_paths_summary(raise_on_missing=False)
print_directory_content(dir=DATA_DIR)


Current Working Directory: /app
Data Path: /app/data/synthetic/10000p
Data Processed Path: /app/data/synthetic/10000p/processed

================= ICARE_RISK WORKSPACE PATHS SUMMARY =================
  [✓] Found    📁 DIR  | Project Root          : /app
  [✓] Found    📁 DIR  | Config Directory      : /app/configs
  [✓] Found    📁 DIR  | Data Directory        : /app/data
  [✓] Found    📁 DIR  | Output Directory      : /app/outputs
  [✓] Found    📄 FILE | Database Config       : /app/configs/generate_db.yaml
  [✓] Found    📄 FILE | Code Search Config    : /app/configs/code_search.yaml
  [✓] Found    📄 FILE | Schema YAML           : /app/configs/schema.yaml
  [✓] Found    📄 FILE | Analysis YAML         : /app/configs/analysis.yaml
  [✓] Found    📄 FILE | Phenotypes YAML       : /app/configs/phenotypes.yaml
  [✓] Found    📄 FILE | Lookup 3GCR           : /app/configs/lookups/3GCR.csv
  [✓] Found    📄 FILE | Lookup 3GCS           : /app/configs/lookups/3GCS.csv
✨ All workspace paths successf

PosixPath('/app/data/synthetic/10000p')

In [18]:
# Convert to parquets if needed
#from icare_risk.utils.conversion import convert
#convert(DATA_DIR, 'csv_to_parquet')

Converting: icare_episodes_anon.csv -> icare_episodes_anon.parquet
Converting: icare_episodes_diagnosis_anon.csv -> icare_episodes_diagnosis_anon.parquet
Converting: icare_demographic_anon.csv -> icare_demographic_anon.parquet
Converting: icare_microbiology_anon.csv -> icare_microbiology_anon.parquet
Converting: icare_problems_anon.csv -> icare_problems_anon.parquet
Converting: icare_pharmacy_prescribing_anon.csv -> icare_pharmacy_prescribing_anon.parquet
Converting: icare_pharmacy_administration_anon.csv -> icare_pharmacy_administration_anon.parquet
Converting: icare_vital_signs_anon.csv -> icare_vital_signs_anon.parquet
Converting: icare_pathology_blood_anon.csv -> icare_pathology_blood_anon.parquet
Converting: table1_demographics.csv -> table1_demographics.parquet
Successfully converted 10 file(s).


## Load source tables

In [13]:
# Load data
df_episodes = pd.read_parquet(DATA_DIR / "icare_episodes_anon.parquet")                # parse_dates=["ADMISSION_DATE"],
df_problems = pd.read_parquet(DATA_DIR / "icare_problems_anon.parquet")                # parse_dates=["PROBLEM_DT_TM"],
df_diagnoses = pd.read_parquet(DATA_DIR / "icare_episodes_diagnosis_anon.parquet")     # parse_dates=["DIAGNOSIS_DATE"],
df_prescribing = pd.read_parquet(DATA_DIR / "icare_pharmacy_prescribing_anon.parquet") # parse_dates=["ORDER_DT_TM"],

# Convert string columns back to proper date-times
df_episodes["ADMISSION_DATE"] = pd.to_datetime(df_episodes["ADMISSION_DATE"])
df_problems["PROBLEM_DT_TM"] = pd.to_datetime(df_problems["PROBLEM_DT_TM"])
df_diagnoses["DIAGNOSIS_DATE"] = pd.to_datetime(df_diagnoses["DIAGNOSIS_DATE"])
df_prescribing["ORDER_DT_TM"] = pd.to_datetime(df_prescribing["ORDER_DT_TM"])

# Show info
print(f"Episodes:    {df_episodes.shape}")
print(f"Problems:    {df_problems.shape}")
print(f"Diagnoses:   {df_diagnoses.shape}")
print(f"Prescribing: {df_prescribing.shape}")

# Brief check
check_df(
    df_episodes,
    "Episodes input",
    columns=[
        "SUBJECT",
        "SPELL_IDENTIFIER",
        "ADMISSION_DATE",
        "DISCHARGE_DATE",
    ],
    n=5,
)

Episodes:    (10000, 11)
Problems:    (30012, 6)
Diagnoses:   (29962, 12)
Prescribing: (69621, 16)

CHECK: Episodes input
Shape: 10,000 rows x 11 columns

Preview:


,SUBJECT,SPELL_IDENTIFIER,ADMISSION_DATE,DISCHARGE_DATE
0,10001,7867287,2023-02-12,2023-02-26
1,10002,6551735,2022-11-16,2022-12-01
2,10003,4441228,2022-05-12,2022-05-27
3,10004,5593202,2023-10-11,2023-10-30
4,10005,6532182,2023-08-14,2023-08-18


## Configure domains and phenotypes

In [14]:
domain_registry = {
    "prescribing": {
        "df": df_prescribing,
        "subject_col": "SUBJECT",
        "time_col": "ORDER_DT_TM",
        "target_col": "MEDICATION_NAME",
        "match_type": "contains",
    },
    "problems": {
        "df": df_problems,
        "subject_col": "SUBJECT",
        "time_col": "PROBLEM_DT_TM",
        "target_col": "PROBLEM_CODE",
    },
    "diagnosis": {
        "df": df_diagnoses,
        "subject_col": "SUBJECT",
        "time_col": "DIAGNOSIS_DATE",
        "sources": [
            {"target_col": "DIAGNOSIS_CODE_SNOMED", "match_type": "exact"},
            {"target_col": "DIAGNOSIS_CODE_ICD",    "match_type": "prefix"},
        ],
    },
}

all_phenotypes = load_pkg_yaml("config/icare/phenotypes.yaml")
charlson = {k:v for k,v in all_phenotypes.items() if k.startswith('charlson_')}
phenotypes = all_phenotypes  # Replace with a filtered dictionary for a smaller run.

print(f"Configured phenotypes: {len(phenotypes):,}")

Configured phenotypes: 66


## Extract and window events

In [15]:
# 1. Extract raw historical events
df_events = run_vectorized(
    domains=domain_registry,
    phenotypes=phenotypes,
    verbose=1,
)

# 2. Create index_admission column
df_episodes["index_admission"] = pd.to_datetime(
    df_episodes["ADMISSION_DATE"].astype(str) + " "
    + df_episodes["ADMISSION_TIME"].astype(str),
    errors="coerce",
)

# 3. Build the window configuration mapping from your phenotypes YAML dictionary
window_configs = {}
for batch in parse_phenotypes_to_batches(phenotypes):
    for sub_event_name, config in batch["configs"].items():
        if config.get("window") is not None:
            window_configs[sub_event_name] = config["window"]

# 4. Filter events given the episode and the windows
df_events_windowed = evaluate_temporal_windows(
    episodes_df=df_episodes,
    events_df=df_events,
    window_configs=window_configs,
    subject_col="SUBJECT",
    admission_col="index_admission",
    event_col="event_name",
    event_time_col="event_dt_tm",
    prefix="hx_",
)

# 5. Aggregate component-level binary indications into parent clinical phenotypes
df_aggregated = aggregate_composite_phenotypes(
    df=df_events_windowed,
    phenotype_configs=phenotypes,
    prefix="",
    drop_components=True,
)

print(f"Raw events:       {df_events.shape}")
print(f"Windowed events:  {df_events_windowed.shape}")
print(f"Aggregated output:{df_aggregated.shape}")


--- Running Vectorized Bench ---

--- 1. Parsing Phenotypes YAML ---
Generated 7 execution batch(es).

--- 2. Executing Batches ---

--- DEBUG BATCH 1 (Source 1) [problems -> PROBLEM_CODE] ---
Match Type: exact, Case Sensitive: False
YAML is looking for (first 5):
  - charlson_hx_mi__comp0
  - charlson_hx_mi__comp1
  - charlson_hx_chf__comp0
  - charlson_hx_chf__comp1
  - charlson_hx_pvd__comp0
DataFrame top 5 most common values (with counts):
  - 77386006: 4,498 occurrence(s)
  - none: 1,428 occurrence(s)
  - 38341003: 1,129 occurrence(s)
  - 38907003: 658 occurrence(s)
  - 195967001: 549 occurrence(s)
-> Matched 2393 record(s) using column 'PROBLEM_CODE' with match_type='exact'.

--- DEBUG BATCH 2 (Source 1) [prescribing -> MEDICATION_NAME] ---
Match Type: contains, Case Sensitive: False
YAML is looking for (first 5):
  - charlson_hx_chf__comp2
  - charlson_hx_diabetes_uncomp__comp2
  - gavaghan_hx_prior_antibiotics_90d__comp0
  - pitt_hypo_score_rule__comp1
DataFrame top 5 most com

In [16]:
# Local Libraries
from icare_risk.clinphen.scores.gavaghan import compute_gavaghan_score
from icare_risk.clinphen.scores.holmgren import compute_holmgren_score
from icare_risk.clinphen.scores.charlson import compute_charlson_score
from icare_risk.clinphen.scores.jones import compute_jones_score
from icare_risk.clinphen.scores.tumbarello import compute_tumbarello_score
from icare_risk.clinphen.scores.kim import compute_kim_score

df_aggregated['charlson_score'] = compute_charlson_score(df_aggregated, verbose=0)
df_aggregated['gavaghan_score'] = compute_gavaghan_score(df_aggregated, verbose=0)
df_aggregated['holmgren_score'] = compute_holmgren_score(df_aggregated, verbose=0)
df_aggregated['jones_score'] = compute_jones_score(df_aggregated, verbose=0)
df_aggregated['tumbarello_score'] = compute_tumbarello_score(df_aggregated, verbose=0)
df_aggregated['kim_score'] = compute_kim_score(df_aggregated, verbose=0)

## Save artifacts

In [17]:
from icare_risk.clinphen.utils.io import save_phenotype_artifacts

save_phenotype_artifacts(
    df_raw=df_events,
    df_windowed=df_events_windowed,
    df_aggregated=df_aggregated,
    output_dir=DATA_PROC_DIR,
    prefix="02a_derived_",
)


--- Saving Phenotype Artifacts ---
 Saved Raw extracted matching events (142,636 rows)
   └─ Path: /app/data/synthetic/10000p/processed/02a_derived_events_extracted_raw.parquet
 Saved Window-filtered events (10,000 rows)
   └─ Path: /app/data/synthetic/10000p/processed/02a_derived_events_extracted_windowed.parquet
 Saved Aggregated patient/spell phenotypes (10,000 rows)
   └─ Path: /app/data/synthetic/10000p/processed/02a_derived_phenotypes_aggregated.parquet
-----------------------------------



## Visually debugging with total counts

In [87]:
def event_summary_v1(df_events):
    return (
        df_events.groupby("event_name")
        .agg(
            total_records=("SUBJECT", "count"),
            unique_patients=("SUBJECT", "nunique"),
        )
        .reset_index()
        .sort_values("unique_patients", ascending=False)
    ).to_string(index=False)

def event_summary_v2(df_events_windowed):
    # 1. Identify all columns starting with 'hx_'
    hx_cols = [col for col in df_events_windowed.columns if col.startswith("hx_")]

    # 2. Group by SUBJECT, check if any row per patient is True/1, then sum
    return (
        df_events_windowed[["SUBJECT"] + hx_cols]
        .groupby("SUBJECT")
        .any()  # True if the patient has a 1 in at least one episode
        .sum()  # Counts total unique patients with True per column
        .reset_index(name="patient_count")
        .rename(columns={"index": "phenotype"})
        .sort_values("patient_count", ascending=False)
    ).to_string(index=False)

In [29]:
print(event_summary_v1(df_events))

                              event_name  total_records  unique_patients
      charlson_hx_diabetes_uncomp__comp2          19884             8468
     kim_hx_prior_antibiotics_90d__comp0          20102             8419
    kim_hx_recent_antibiotics_90d__comp1          20102             8419
gavaghan_hx_prior_antibiotics_90d__comp1          20102             8419
     tumbarello_hx_recent_abx_90d__comp0          20102             8419
   jones_hx_prior_antibiotics_30d__comp1          20102             8419
    kim_hx_recent_antibiotics_90d__comp0          10098             6238
   jones_hx_prior_antibiotics_30d__comp0          10098             6238
      charlson_hx_diabetes_uncomp__comp0            828              803
        charlson_hx_renal_mod_sev__comp0            256              252
               charlson_hx_stroke__comp0            190              188
            charlson_hx_pulmonary__comp0            175              174
             charlson_hx_dementia__comp0           

In [88]:
print(event_summary_v2(df_events_windowed))

                                  phenotype  patient_count
      hx_charlson_hx_diabetes_uncomp__comp2           5552
    hx_kim_hx_recent_antibiotics_90d__comp1           1131
     hx_tumbarello_hx_recent_abx_90d__comp0           1119
hx_gavaghan_hx_prior_antibiotics_90d__comp1           1119
     hx_kim_hx_prior_antibiotics_90d__comp0           1119
    hx_kim_hx_recent_antibiotics_90d__comp0            588
      hx_charlson_hx_diabetes_uncomp__comp0            418
   hx_jones_hx_prior_antibiotics_30d__comp1            393
   hx_jones_hx_prior_antibiotics_30d__comp0            199
        hx_charlson_hx_renal_mod_sev__comp0            127
            hx_charlson_hx_pulmonary__comp0             90
               hx_charlson_hx_stroke__comp0             88
             hx_charlson_hx_dementia__comp0             61
                  hx_charlson_hx_chf__comp0             58
                   hx_charlson_hx_mi__comp0             55
                  hx_charlson_hx_pud__comp0             

In [89]:
check_value_counts(
    df_events, "event_name", name="Events by phenotype component", n=20,
)

check_value_counts(
    df_events, "domain", name="Events by domain", n=20,
)


Value check: Events by phenotype component


,count
event_name,
gavaghan_hx_prior_antibiotics_90d__comp1,20021
jones_hx_prior_antibiotics_30d__comp1,20021
kim_hx_recent_antibiotics_90d__comp1,20021
tumbarello_hx_recent_abx_90d__comp0,20021
kim_hx_prior_antibiotics_90d__comp0,20021
charlson_hx_diabetes_uncomp__comp2,19848
kim_hx_recent_antibiotics_90d__comp0,10145
jones_hx_prior_antibiotics_30d__comp0,10145
charlson_hx_diabetes_uncomp__comp0,844



Value check: Events by domain


,count
domain,
prescribing,140243
problems,2393
